# BIND2 Appendices — Model variants

**A** VDM vs Flow-Matching (`vdm_fm.npz`); **B** observable-conditioned FM (`observables.npz`).

In [ ]:

import sys
sys.path.insert(0, '/mnt/home/mlee1/vdm_bind2/tools/paper_cache')
import os, pickle
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import paper_config as C

CACHE = C.CACHE_DIR
def L(name):
    p = CACHE / name
    if name.endswith('.pkl'):
        return pickle.load(open(p, 'rb'))
    return dict(np.load(p, allow_pickle=False))

FIG_DIR = Path('paper_figures'); FIG_DIR.mkdir(exist_ok=True)
def save_fig(fig, name, ext=('pdf', 'png')):
    for e in ext:
        fig.savefig(FIG_DIR / f'{name}.{e}', dpi=300, bbox_inches='tight')
    print('  saved', name)

try:
    import scienceplots  # noqa: F401
    plt.style.use(['science', 'notebook'])
except Exception:
    pass
# plt.rcParams.update({'font.size': 10, 'font.family': 'serif', 'mathtext.fontset': 'cm',
#                      'figure.dpi': 110, 'savefig.dpi': 300, 'axes.grid': False})

SUITE_COLORS = C.SUITE_COLORS; SUITE_DISPLAY = C.SUITE_DISPLAY
MASS_CH = C.MASS_CHANNELS; CH_DISPLAY = C.CH_DISPLAY
BIN_LABELS = C.MASS_BIN_LABELS; N_BINS = C.N_MASS_BINS
PARAM_LABELS = C.PARAM_LABELS
# Trained-regime restriction: the paper only uses halos with M200c >= 1e13 (the
# training cut). Lower bins exist in the cache but are never plotted.
MIN_LOG_M200 = 13.0
BINS = [b for b in range(N_BINS) if C.MASS_EDGES[b] >= MIN_LOG_M200]
BIN_CMAP = np.zeros((N_BINS, 4))
BIN_CMAP[BINS] = plt.cm.viridis(np.linspace(0.1, 0.9, len(BINS)))
print('Cache :', CACHE)
print('Model :', C.MODEL_TAG, '| suites', {s: 0 for s in C.SUITES})
print('Files :', sorted(p.name for p in CACHE.glob("*.pkl")) + sorted(p.name for p in CACHE.glob("*.npz")))


## Appendix A · VDM vs Flow-Matching

Same UNet, two formulations. VDM is sharper at small scales (mass↔structure tension); FM is smoother/mass-correct.

In [ ]:

vf = L('vdm_fm.npz')
def lg(im): p=im[im>0]; return np.log10(np.clip(im, p.min() if len(p) else 1e-30, None))
fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))
ax[0].imshow(lg(vf['fm_gas']), cmap='magma'); ax[0].set_title('FM (fm_redshift) Gas'); ax[0].set_xticks([]); ax[0].set_yticks([])
ax[1].imshow(lg(vf['vdm_gas']), cmap='magma'); ax[1].set_title('VDM Gas'); ax[1].set_xticks([]); ax[1].set_yticks([])
ax[2].loglog(vf['kf'], vf['pf'], label='FM'); ax[2].loglog(vf['kv'], vf['pv'], label='VDM')
ax[2].set(xlabel='$k$ [$h$/Mpc]', ylabel='Gas $P(k)$ [patch]', title=f"Stacked patch P(k) — VDM/FM (k>20) = {float(vf['highk_ratio']):.2f}"); ax[2].legend()
save_fig(fig, 'figA_vdm_vs_fm'); plt.show()


## Appendix B · Observable-conditioned emulation (M+Y+Tx)

Condition on aperture-integrated R200 observables instead of the 35 parameters.

In [ ]:

ob = L('observables.npz')
def lg(im): p=im[im>0]; return np.log10(np.clip(im, p.min() if len(p) else 1e-30, None))
fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))
ax[0].imshow(lg(ob['truth_gas'][0]), cmap='magma'); ax[0].set_title('truth Gas'); ax[0].set_xticks([]); ax[0].set_yticks([])
ax[1].imshow(lg(ob['gen_gas'][0]), cmap='magma'); ax[1].set_title('BIND2 | M+Y+Tx Gas'); ax[1].set_xticks([]); ax[1].set_yticks([])
mt, mg = ob['truth_gas_mass'], ob['gen_gas_mass']
ax[2].loglog(mt, mg, '.', ms=8); lim=[mt.min(), mt.max()]; ax[2].plot(lim, lim, 'k--')
ax[2].set(xlabel='truth Gas mass [patch]', ylabel='obs-conditioned Gas mass', title='Gas recovery (M+Y+Tx)')
save_fig(fig, 'figB_observable_conditioned'); plt.show()
